In [1]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime

# 取得今天的日期，格式化成 YYYYMMDD
today_date = datetime.today().strftime('%Y%m%d')

print(today_date)  # 例如: 20250317

20250324


In [2]:
# aid = 3 : NBA
input_date = '20250210'
user_agent = ''
headers = {'User-Agent': user_agent}
url = f'https://www.playsport.cc/livescore.php?aid=3&gamedate={input_date}&mode=11'
response = requests.get(url, headers=headers)

# 確認是否成功
if response.status_code == 200:
    print('連線成功')
else:
    print('連線失敗')

連線成功


In [3]:
html_source = response.text
soup = BeautifulSoup(html_source, 'html.parser')

In [19]:
# aid = 3 : NBA
input_date = '20250321'
user_agent = ''
headers = {'User-Agent': user_agent}
url = f'https://www.playsport.cc/livescore.php?aid=3&gamedate={input_date}&mode=11'
response = requests.get(url, headers=headers)

# 確認是否成功
if response.status_code == 200:
    print('連線成功')
else:
    print('連線失敗')

html_source = response.text
soup = BeautifulSoup(html_source, 'html.parser')

# 找到比賽容器
livescore_container = soup.find('div', id='livescoreContainer')

# 確認是否有比賽數據
if not livescore_container or '暫無資料' in livescore_container.text:
    print("沒有比賽數據")
    games = []
else:
    games = []
    try:
        # 找到所有比賽區塊
        for game in soup.find_all('div', class_='js-gameOnbox'):
            team_a = game.get('data-namea', '未知隊伍').strip()  # 客隊
            team_h = game.get('data-nameh', '未知隊伍').strip()  # 主隊

            score_a = game.find('td', class_='big_score', id=lambda x: x and x.endswith('_asr_big'))
            score_h = game.find('td', class_='big_score', id=lambda x: x and x.endswith('_hsr_big'))
            score_a = score_a.text.strip() if score_a else '0'
            score_h = score_h.text.strip() if score_h else '0'

            # 每局得分
            innings_a = []
            innings_h = []
            for i in range(1, 6):  # NBA最多5局
                inning_a = game.find('td', id=lambda x: x and x.endswith(f'_as{i}'))
                inning_h = game.find('td', id=lambda x: x and x.endswith(f'_hs{i}'))
                innings_a.append(inning_a.text.strip() if inning_a else '')
                innings_h.append(inning_h.text.strip() if inning_h else '')

            # R, T, L (總得分、加總、分差)

            total_r_a = game.find('td', id=lambda x: x and x.endswith('_asr'))
            
            total_r_h = game.find('td', id=lambda x: x and x.endswith('_hsr'))
            

            total_t = game.find('td', id=lambda x: x and x.endswith('_ts'))
            total_l = game.find('span', id=lambda x: x and x.startswith('js-leadingpoint-'))

            

            game_data = {
                'team_a': team_a,
                'team_h': team_h,
                'score_a': total_r_a.text.strip() if total_r_a else '0',
                'score_h': total_r_h.text.strip() if total_r_h else '0',
                'innings': {
                    'team_a': innings_a,                
                    'final_a': total_r_a.text.strip() if total_r_a else '0',
                    'team_h': innings_h,
                    'final_h': total_r_h.text.strip() if total_r_h else '0'                    
                },
                'summary': {
                    'total_score': total_t.text.strip() if total_t else '0' ,                
                    'leading_point': total_l.text.strip() if total_l else 'N/A',
                }
            }

            games.append(game_data)

    except Exception as e:
        print(f"發生錯誤: {e}")
        games = []

連線成功


In [20]:
games

[{'team_a': '籃網',
  'team_h': '溜馬',
  'score_a': '99',
  'score_h': '105',
  'innings': {'team_a': ['28', '24', '19', '20', '8'],
   'final_a': '99',
   'team_h': ['23', '19', '25', '24', '14'],
   'final_h': '105'},
  'summary': {'total_score': '204', 'leading_point': '主贏6'}},
 {'team_a': '尼克',
  'team_h': '黃蜂',
  'score_a': '98',
  'score_h': '115',
  'innings': {'team_a': ['19', '25', '28', '26', ''],
   'final_a': '98',
   'team_h': ['27', '27', '31', '30', ''],
   'final_h': '115'},
  'summary': {'total_score': '213', 'leading_point': '主贏17'}},
 {'team_a': '公牛',
  'team_h': '國王',
  'score_a': '128',
  'score_h': '116',
  'innings': {'team_a': ['28', '27', '37', '36', ''],
   'final_a': '128',
   'team_h': ['39', '25', '25', '27', ''],
   'final_h': '116'},
  'summary': {'total_score': '244', 'leading_point': '客贏12'}},
 {'team_a': '暴龍',
  'team_h': '勇士',
  'score_a': '114',
  'score_h': '117',
  'innings': {'team_a': ['30', '31', '31', '22', ''],
   'final_a': '114',
   'team_h': [

In [7]:
soup

<!DOCTYPE html>

<html lang="zh-Hant">
<head>
<meta content="notranslate" name="google"/>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="telephone=no" name="format-detection"/>
<meta content="webServer2" name="hostname"/>
<meta content="台灣運彩,討論區,賽事討論,運動討論,運動彩券討論,運彩討論,運彩分析,運動分析,運動彩券分析,運動彩券,運動彩,運彩,玩運彩,玩韻彩,完運彩,完韻采,運彩朋友圈,賽事預測,美國職棒,日本職棒,中華職棒,NBA,MLB,足球,韓國職棒,墨西哥棒球,韓國女籃,日本職籃,中國職籃,韓國職籃,俄羅斯冰球,美式足球,網球,KBO,KHL,NHL,NPB,CPBL,NFL,WNBA,世足,世界盃,即時比分" name="keywords">
<meta content="全球最快中文即時比分，提供MLB美國職棒、NBA、日本職棒、中華職棒、韓棒、P+、SBL、韓國職籃、日本職籃、中國職籃...等，運彩迷不能錯過。" name="description">
<meta content="即時比分" property="og:title"/>
<meta content="sport" property="og:type"/>
<meta content="https://www.playsport.cc/livescore.php" property="og:url"/>
<meta content="https://www.playsport.cc/includes/images/playsport_square_big.png" property="og:image"/>
<meta content="玩運彩" property="og:site_name"/>
<meta content="100003206578531" property="fb:admins"/>
<link href="./includes/images/sta